# 03 HOD And Population Pipeline

This notebook organizes HOD, LRG/ELG, satellite radial-distribution, and merger
population analyses.


In [ ]:
import os
from pathlib import Path
import sys

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "ia_analysis").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ia_analysis checkout.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RUNTIME_DIR = PROJECT_ROOT / ".notebook_runtime"
RUNTIME_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(RUNTIME_DIR / "matplotlib"))
os.environ.setdefault("IPYTHONDIR", str(RUNTIME_DIR / "ipython"))

print("Project root:", PROJECT_ROOT)


In [ ]:
exports = PROJECT_ROOT / "src" / "ia_analysis" / "notebook_pipelines" / "exports"
for name in [
    "hod_lrg_elg_nb.py",
    "hod_data_nb.py",
    "hod_measure_lrg_elg_nb.py",
    "maset_satellite_radial_distribution_nb.py",
    "maset_satellite_radial_distribution_compare_nb.py",
    "merger_align_nb.py",
    "merger_stripping_nb.py",
]:
    print(exports / name)


## Suggested Workflow

1. Load a sample from a global HDF5 file or from the original FoF/Subhalo catalogs.
2. Split by stellar mass, SFR, central/satellite status, and host mass.
3. Produce HOD tables, satellite radial profiles, and merger diagnostics.
4. Write tables to `outputs/tables` and figures to `outputs/figures/hod_population`.


## Legacy Notebook Function Coverage

The functions below came from the former notebooks mapped to this workflow.
They remain visible here for compatibility and can be inspected without
executing the old notebooks' top-level data-loading cells.

### Pipeline and analysis functions

- `hod_data_nb.py`: `flag_display`, `flag_color`, `format_z_title`, `output_safe_name`, `_decode_attr`, `_safe_group_name`, `canonical_flag`, `_parse_file_metadata_from_name`, `read_file_metadata`, `discover_hod_files`, `_find_key`, `_find_region_key`, `list_hod_structure`, `get_hod_path`, `read_hod_curve`, `_profile_label_from_attrs`, `read_profile_curves`, `profile_mass_bin_order`, `mass_bin_style_map`
- `hod_lrg_elg_nb.py`: `_try_load_fof`, `_choose_first_sub_indexing`, `_periodic_delta`, `_compute_subhalo_radius_to_host`, `load_sim_direct`, `safe_ssfr`, `_redshift_dependent_cut`, `elg_logssfr_cut`, `lrg_mstar_cut_msun`, `select_lrg`, `select_elg`, `make_analysis_specs`, `compute_occupation_counts_from_sim`, `bin_occupation_curve`, `_finite_log10_range`, `_selected_stellar_to_host_summary`, `_selected_r_over_r200c_summary`, `build_curves_for_snapshot`, `smooth_curve_logx`, `_ensure_curve_data_dirs`, `_safe_name`, `_curve_hdf5_path`, `_to_python_scalar`, `_jsonable_row`, `_as_1d`, `_replace_group`, `_write_dataset`, `_write_attrs`, `_write_json_dataset`, `_write_common_meta`, `save_hod_curve_data`, `save_radial_profile_curve_data`, `inspect_curve_hdf5`, `_read_json_dataset`, `_read_dataset_if_exists`, `_curve_hdf5_has_group`, `all_curve_hdf5_available`, `load_hod_curve_data_from_hdf5`, `load_radial_profile_curve_data_from_hdf5`, `get_or_build_hod_curves`, `get_or_build_radial_profiles`, `_try_load_tng_fof`, `load_tng_direct`, `build_tng_curves_for_snapshot`, `_radial_mass_bin_label`, `compute_radial_profile_from_sim`, `build_radial_profiles_for_snapshot`, `_smooth_profile_x`
- `hod_measure_lrg_elg_nb.py`: `format_catalog_path`, `find_candidate_files`, `infer_format`, `read_hdf5_as_dataframe`, `read_table`, `get_column`, `table_shape`, `read_halo_catalog`, `read_galaxy_catalog`, `inspect_catalogue`, `convert_host_indices`, `central_mask_from_config`, `count_occupations`, `hod_from_counts`, `measure_one`, `measure_all`, `_safe_hod_xy`, `add_environment_quantiles`, `hod_from_counts_by_environment`, `sanity_check_counts`
- `maset_satellite_radial_distribution_nb.py`: `get_snap_dict`, `get_arr`, `build_mask_from_limits`, `make_analysis_specs`, `compute_radial_profile`, `build_profiles_for_snapshot`
- `maset_satellite_radial_distribution_compare_nb.py`: `get_snap_dict`, `get_arr`, `build_mask_from_limits`, `make_analysis_specs`, `compute_radial_profile`, `build_profiles_for_snapshot`, `build_profile_cube`, `_setup_compare_axes`

### Plotting functions

- `hod_data_nb.py`: `save_figure`, `maybe_close`, `add_axis_colorbar`, `apply_hod_axis_style`, `model_legend_handles`, `component_legend_handles`, `add_two_row_legend`, `add_component_legend`, `plot_hod_curve`, `plot_model_comparison_grid`, `plot_redshift_evolution_by_model`, `apply_profile_axis_style`, `profile_mass_bin_legend_handles`, `add_profile_two_row_legend`, `add_profile_mass_legend`, `plot_profile_curve`, `plot_profile_model_comparison_grid`, `plot_profile_redshift_evolution_by_model`, `plot_one_profile_set`, `plot_all_profile_comparisons`, `plot_one_hod_set`, `plot_all_hod_comparisons`, `plot_all_comparisons`
- `hod_lrg_elg_nb.py`: `_plot_one_curve`, `plot_hod_grid`, `plot_radial_profile_grid`
- `hod_measure_lrg_elg_nb.py`: `preview_configured_paths`, `plot_components_for_model`, `plot_models_for_component`, `plot_all_hod`, `plot_environment_hod`
- `maset_satellite_radial_distribution_nb.py`: `plot_model_by_sample_grid`
- `maset_satellite_radial_distribution_compare_nb.py`: `plot_original_grid`, `plot_redshift_comparison_grid`, `plot_gravity_comparison_grid`


In [ ]:
from IPython.display import Code, display
from ia_analysis.notebook_pipelines import legacy_catalog

LEGACY_EXPORTS = ('hod_data_nb.py', 'hod_lrg_elg_nb.py', 'hod_measure_lrg_elg_nb.py', 'maset_satellite_radial_distribution_nb.py', 'maset_satellite_radial_distribution_compare_nb.py')
legacy_manifest = legacy_catalog.manifest(LEGACY_EXPORTS)

print("Pipeline definitions:", len(legacy_manifest["pipeline"]))
print("Plotting definitions:", len(legacy_manifest["plotting"]))

def show_legacy_source(export, name, occurrence=1):
    """Display a preserved function or class from a former notebook."""
    text = legacy_catalog.source(export, name, occurrence=occurrence)
    display(Code(text, language="python"))
    return text

# Example:
# show_legacy_source(LEGACY_EXPORTS[0], legacy_manifest["pipeline"][0].name)
